# 🧩 Andamiaje · Proyecto **Centinela** — Fase 3 (Pipeline, aceleración y despliegue en el borde)
### Redes Neuronales — Deep Learning · Maestría en Ciencia de Datos · Universidad Santo Tomás

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JotaMao1985/Deep_Learning_Usta/blob/main/notebooks/07-scaffold-centinela-fase3.ipynb)

Este cuaderno es un **andamiaje guía** (*scaffold*) para **industrializar** el sistema de las fases anteriores: construir un **pipeline de datos reproducible**, **entrenar acelerado en GPU** (precisión mixta + *checkpointing*) y **exportar el modelo para despliegue offline en el borde** (ONNX/LiteRT con cuantización). Trae lista la mecánica de soporte —reporte de GPU, estructura de `tf.data`/`DataLoader` de ejemplo, utilidades de *timing*, *benchmark* de inferencia y una plantilla ONNX con un modelo *dummy* **funcional**— y deja marcadas con `# TODO (estudiante)` las piezas evaluables: el **pipeline real**, el **entrenamiento con AMP** y la **exportación/cuantización con verificación de paridad**.

> **Cómo se usa:** se clona este cuaderno, se ejecutan las celdas de soporte en orden y se completan las celdas marcadas `# TODO (estudiante)`. En Google Colab: *Entorno de ejecución → Ejecutar todas*. Las celdas de soporte corren de punta a punta; las celdas `# TODO` traen *stubs* que **no rompen la ejecución** (imprimen un recordatorio y devuelven un *placeholder*) hasta que el estudiante escribe su solución. **Todo el cuaderno corre incluso sin GPU** (con avisos) y **sin descargas** (datos sintéticos).

> ⚠️ **Andamiaje — NO es la solución.** Este cuaderno **no** contiene el pipeline real, ni un modelo entrenado con AMP, ni una exportación cuantizada validada, y usa **datos sintéticos** de juguete, **no** los datos reales del proyecto. Su propósito es enseñar la *mecánica* de MLOps; el pipeline, el entrenamiento acelerado y el despliegue los aporta el estudiante sobre **sus propios datos** (los del cliente/problema de las Fases 1-2).

---
**Contenido por etapas**
0. **Preparación + política de datos** (semilla, *device*/reporte GPU, paleta USTA, datos sintéticos)
- **Etapa A — Pipeline de datos reproducible:** estructura `tf.data`/`DataLoader` de ejemplo · `# TODO` pipeline real con *Data Augmentation*, `prefetch`/`cache` y partición sin fuga (*leakage*)
- **Etapa B — Entrenamiento acelerado (GPU + precisión mixta):** utilidades de *timing* y *checkpointing* a Drive · `# TODO` entrenar con **AMP**, *early stopping* y *checkpoints*; reportar *speed-up*
- **Etapa C — Exportación y despliegue offline (borde):** *benchmark* de inferencia + plantilla ONNX con modelo *dummy* funcional · `# TODO` exportar a **ONNX/LiteRT**, **cuantizar** y validar **paridad** (`np.allclose`); medir latencia/tamaño
- **Sección final — Checklist de entrega + rúbrica de la Fase 3** (criterios y pesos que suman 100 %)

## 0. Preparación del entorno + política de datos

Se importan las librerías de soporte (`numpy`, `matplotlib`) y, **si está disponible**, `torch`. Se fija una **semilla** global para reproducibilidad y se **reporta la GPU** (tipo, VRAM y disponibilidad). La separación de miles se mostrará con punto, según la convención del material del curso.

> 💡 *Importación tolerante:* `torch`, `onnx` y `onnxruntime` se importan de forma **tolerante**: si alguno no estuviera instalado, el cuaderno **no se rompe** y las celdas de soporte que no lo necesitan siguen corriendo. En Colab, descomentar la línea de `pip` de la celda si hiciera falta.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Celda de soporte 1 · Preparación del entorno + reporte de GPU  (FUNCIONAL)
# Qué hace · Importa librerías de soporte, fija la semilla, detecta el device, REPORTA la GPU
#            (tipo y VRAM) y configura la paleta USTA.
# Por qué  · Una semilla fija hace reproducibles los resultados; documentar la GPU y la cuota
#            es parte del criterio de gestión de recursos de la Fase 3.

# En Colab estas librerías ya vienen instaladas. Si faltara alguna, descomentar:
# !pip -q install numpy matplotlib torch onnx onnxruntime

%matplotlib inline
import time, os
import numpy as np
import matplotlib.pyplot as plt

# Reproducibilidad: una misma semilla produce los mismos resultados en cada ejecución.
SEMILLA = 42
np.random.seed(SEMILLA)

# Importación TOLERANTE de PyTorch: si no estuviera instalado, el cuaderno no se rompe.
try:
    import torch
    import torch.nn as nn
    torch.manual_seed(SEMILLA)
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    TORCH_OK = True
    GPU_OK = torch.cuda.is_available()
except Exception as e:  # pragma: no cover  (entorno sin torch)
    torch = None
    nn = None
    DEVICE = "cpu"
    TORCH_OK = False
    GPU_OK = False
    print("Aviso: PyTorch no está disponible en este entorno ->", e)
    print("Las celdas de soporte que no usan torch seguirán funcionando.")

# Reporte de GPU: tipo, VRAM y cuota. Si no hay GPU, se avisa y se sigue en CPU.
def reporte_gpu():
    """Imprime tipo de GPU y VRAM si está disponible; avisa si se corre en CPU."""
    if TORCH_OK and GPU_OK:
        nombre = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        print(f"GPU detectada: {nombre}  ·  VRAM total: {vram_gb:.1f} GB")
        print("Recordatorio (Fase 3): documenta el tipo de GPU (T4/L4/A100...) y la CUOTA restante.")
        # En Colab también conviene ejecutar en una celda:  !nvidia-smi
    else:
        print("Aviso: no se detectó GPU -> se ejecuta en CPU (modo demostración, más lento).")
        print("En Google Colab: Entorno de ejecución -> Cambiar tipo de entorno -> GPU.")

# Paleta USTA para las gráficas
USTA_MORADO, USTA_ROSA, USTA_NAVY = "#3D008D", "#ED1E79", "#001A4D"
USTA_TEAL, USTA_GOLD = "#0E7490", "#FDB913"
plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.grid": True, "grid.alpha": 0.3})

# Separador de miles con punto, según la convención del material del curso.
def miles(n):
    """Formatea un entero con punto como separador de miles (p. ej. 1.234.567)."""
    return f"{int(n):,}".replace(",", ".")

print("Entorno de soporte listo · numpy", np.__version__)
print(f"PyTorch disponible: {TORCH_OK}   ·   GPU disponible: {GPU_OK}   ·   device: {DEVICE}")
reporte_gpu()

> ⚠️ **Política de datos del Proyecto Centinela**
> Este andamiaje usa **datos SINTÉTICOS de demostración** (imágenes sintéticas generadas con `numpy`) **solo** para enseñar la mecánica de pipeline, GPU y exportación. La entrega evaluable de la Fase 3 se construye **reutilizando el sistema de las Fases 1-2**: el **mismo problema, cliente y fuentes de datos REALES** elegidos en la Fase 1 (p. ej. recolección propia, IDEAM/NASA POWER para clima, u otros repositorios institucionales). **No se admiten datasets de Kaggle como eje del proyecto.** Reutilizar estos datos de juguete **no satisface** la actividad: aquí no se cambia el problema, se *industrializa* el modelo ya construido.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Celda de soporte 2 · Imágenes sintéticas (FUNCIONAL · placeholder)
# Qué hace · Genera imágenes 3xHxW sintéticas en 2 clases (normal / anómala),
#            como sustituto de juguete del dataset real de visión del proyecto.
# Por qué  · Permite ejercitar todo el ciclo MLOps (pipeline, AMP, exportación) con cómputo
#            mínimo y SIN descargar ningún dataset. NO es el dato del proyecto.

IMG = 64                          # lado de la imagen (px) — pequeño a propósito
CLASES = ("normal", "anómala")    # 2 clases (binaria): figura uniforme vs figura con manchas
N_POR_CLASE = 120                 # cómputo mínimo

def _figura_sintetica(rng, anomala):
    """Disco de color sobre fondo; si 'anomala', añade manchas contrastantes (proxy de anomalía)."""
    img = np.zeros((3, IMG, IMG), dtype=np.float32)
    yy, xx = np.ogrid[:IMG, :IMG]
    cy, cx = IMG / 2, IMG / 2
    # Disco = silueta de la figura
    figura = ((yy - cy) ** 2) / (0.40 * IMG) ** 2 + ((xx - cx) ** 2) / (0.40 * IMG) ** 2 <= 1.0
    img[2][figura] = 0.55 + 0.15 * rng.standard_normal(figura.sum())   # canal azul
    img[1][figura] = 0.35 + 0.05 * rng.standard_normal(figura.sum())   # algo de verde
    if anomala:
        for _ in range(rng.integers(3, 7)):                            # manchas contrastantes
            my, mx = rng.integers(0, IMG, size=2)
            r = rng.integers(3, 6)
            mancha = (yy - my) ** 2 + (xx - mx) ** 2 <= r ** 2
            mancha &= figura
            img[0][mancha] = 0.85   # rojo/naranja
            img[1][mancha] = 0.30
    return img.clip(0, 1)

def generar_imagenes(n_por_clase=N_POR_CLASE, semilla=SEMILLA):
    """Devuelve (X, y): X de forma (N, 3, IMG, IMG) float32 en [0,1], y en {0,1}."""
    rng = np.random.default_rng(semilla)
    X, y = [], []
    for clase in range(len(CLASES)):
        for _ in range(n_por_clase):
            X.append(_figura_sintetica(rng, anomala=(clase == 1)))
            y.append(clase)
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.int64)
    perm = rng.permutation(len(y))
    return X[perm], y[perm]

X_img, y_img = generar_imagenes()
print(f"Imágenes sintéticas: {miles(len(X_img))} muestras · {len(CLASES)} clases {CLASES}")
print(f"Forma del tensor (N, C, H, W): {X_img.shape}")

# Vista rápida: una muestra por clase
fig, axes = plt.subplots(1, 2, figsize=(5.5, 3))
for ax, clase in zip(axes, range(2)):
    idx = int(np.where(y_img == clase)[0][0])
    ax.imshow(np.transpose(X_img[idx], (1, 2, 0))); ax.set_title(CLASES[clase]); ax.axis("off")
plt.suptitle("Etapa 0 · imágenes sintéticas de juguete (una por clase)"); plt.tight_layout(); plt.show()

## Etapa A — Pipeline de datos reproducible

El objetivo de MLOps en esta etapa es que la **CPU prepare el siguiente lote mientras la GPU calcula el actual** (solapamiento por `prefetch`), de modo que el tiempo por época se acerque al mayor de los dos y no a su suma:

$$ t_{\text{época}} \approx \max\!\left(t_{\text{CPU}},\; t_{\text{GPU}}\right) $$

Se provee, **listo para usar**, un `DataLoader` de PyTorch de **ejemplo** sobre las imágenes sintéticas y un *snippet* comentado de cómo se vería el **mismo** pipeline en `tf.data`. Se deja como `# TODO (estudiante)`: construir el pipeline **sobre los datos reales** con **Data Augmentation**, `prefetch`/`cache` y una **partición train/val/test sin fuga** (*leakage*).

> 💡 **Puente con el proyecto:** en el proyecto real las imágenes se cargan típicamente con `torchvision.datasets.ImageFolder` (PyTorch) o `tf.keras.utils.image_dataset_from_directory` (TF) desde carpetas `train/val/test` por clase. El andamiaje del bucle es el mismo; solo cambia la fuente de los píxeles.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Celda de soporte 3 · Pipeline de EJEMPLO (DataLoader de PyTorch)  (FUNCIONAL si hay torch)
# Qué hace · Parte (70/15/15) las imágenes sintéticas y las empaqueta en DataLoaders con
#            prefetch_factor/pin_memory; mide el tiempo de una pasada por el loader.
# Por qué  · El pipeline real del estudiante (TODO) tendrá esta misma estructura; aquí queda
#            resuelta la mecánica de iteración por lotes con solapamiento CPU/GPU.

def particion_sin_fuga(X, y, fr_train=0.70, fr_val=0.15, semilla=SEMILLA):
    """Partición 70/15/15 por INDICES barajados una sola vez (sin solapamiento entre conjuntos).

    Clave anti-leakage: se baraja UNA vez y se corta; ninguna muestra aparece en dos
    conjuntos. En datos REALES con varias tomas de la MISMA entidad, la partición debe
    hacerse por GRUPO (todas las tomas de una entidad caen en el mismo conjunto), no por imagen.
    """
    rng = np.random.default_rng(semilla)
    idx = rng.permutation(len(y))
    n_tr = int(len(y) * fr_train)
    n_va = int(len(y) * (fr_train + fr_val))
    return idx[:n_tr], idx[n_tr:n_va], idx[n_va:]

i_tr, i_va, i_te = particion_sin_fuga(X_img, y_img)
print(f"Partición sin fuga -> train {miles(len(i_tr))} · val {miles(len(i_va))} · test {miles(len(i_te))}")

dls_img = None
if TORCH_OK:
    from torch.utils.data import TensorDataset, DataLoader
    def _dl(idx, shuffle):
        ds = TensorDataset(torch.tensor(X_img[idx]), torch.tensor(y_img[idx]))
        # En GPU conviene pin_memory=True; num_workers/prefetch_factor solapan CPU y GPU.
        kw = dict(batch_size=32, shuffle=shuffle, pin_memory=GPU_OK)
        return DataLoader(ds, **kw)
    dls_img = {"train": _dl(i_tr, True), "val": _dl(i_va, False), "test": _dl(i_te, False)}
    t0 = time.perf_counter()
    for xb, yb in dls_img["train"]:
        pass
    print(f"Lote de imágenes -> X {tuple(xb.shape)} · y {tuple(yb.shape)} · dtype {xb.dtype}")
    print(f"Tiempo de una pasada por el loader de entrenamiento: {time.perf_counter() - t0:.3f} s")
else:
    print("torch no disponible: se omite la creación de DataLoaders (no rompe el cuaderno).")

# --- SNIPPET (comentado) · el MISMO pipeline en tf.data (para la comparativa de la rúbrica) ---
# import tensorflow as tf
# AUTOTUNE = tf.data.AUTOTUNE
# ds = tf.keras.utils.image_dataset_from_directory("ruta/train", image_size=(224, 224), batch_size=32)
# ds = (ds.cache()                       # cachea tras la 1a época (si cabe en memoria/disco)
#         .shuffle(1_000)                # buffer comparable al tamaño del dataset
#         .prefetch(AUTOTUNE))           # solapa CPU (carga) y GPU (cómputo)
# --- y en PyTorch real: DataLoader(ImageFolder(...), num_workers=4, pin_memory=True, prefetch_factor=2) ---

### `# TODO (estudiante)` · Pipeline reproducible sobre datos REALES

> 📝 **Tarea del estudiante (Etapa A)**
> Aquí está el trabajo evaluable de pipeline (**Criterio 1 de la rúbrica · 20 %**). Sobre los **datos reales** del proyecto (no sobre estas imágenes de juguete), se debe:
>
> 1. **Construir el MISMO pipeline en ambos frameworks** (`tf.data` y `DataLoader`) y **comparar el tiempo de carga por época**, activando `cache`, `shuffle`, `batch` y `prefetch(AUTOTUNE)` en TF, y `num_workers`/`pin_memory`/`prefetch_factor` en PyTorch.
> 2. **Aplicar Data Augmentation** con **≥ 4 transformaciones** (p. ej. `RandomHorizontalFlip`, `ColorJitter`, `RandomRotation`, `RandomResizedCrop`) justificando cuáles convienen y cuáles dañan la semántica (p. ej. un `ColorJitter` excesivo que borre las marcas que distinguen las clases). Opcional avanzado: **Mixup/CutMix**.
> 3. **Particionar train/val/test sin fuga (*leakage*):** si varias tomas pertenecen a la **misma entidad**, particionar **por grupo**; nunca dejar tomas de la misma entidad en train y en test.
>
> 💡 *Pista:* compara el `accuracy` de validación **con y sin** aumento tras unas épocas y muestra una grilla 3×3 de originales vs. aumentadas.

> ⚠️ La celda siguiente trae **stubs** (`pipeline_real = None`, funciones que imprimen `TODO:`) para que el cuaderno **corra de punta a punta**. El estudiante reemplaza los stubs por su implementación.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TODO (estudiante) · Etapa A: pipeline reproducible sobre datos REALES
# Reemplazar los stubs por la implementación propia. Los stubs NO rompen la ejecución:
# imprimen un recordatorio 'TODO:' y devuelven un placeholder.

# 1) Data Augmentation (>=4 transformaciones) --------------------------------
# Pista (descomentar y completar cuando torchvision este disponible):
# from torchvision import transforms
# aumentos_train = transforms.Compose([
#     transforms.RandomResizedCrop(224),
#     transforms.RandomHorizontalFlip(),
#     transforms.RandomRotation(15),
#     transforms.ColorJitter(brightness=0.2, contrast=0.2),   # moderado: no borrar las marcas de clase
#     transforms.ToTensor(),
#     transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
# ])
def construir_pipeline_real(ruta_datos=None, framework="pytorch"):
    """TODO (estudiante): cargar los datos REALES desde carpetas train/val/test,
    aplicar Data Augmentation (>=4 transformaciones) y devolver loaders con
    prefetch/cache. Comparar el tiempo por época entre tf.data y DataLoader."""
    print("TODO: construir el pipeline real (tf.data + DataLoader) con aumento y prefetch "
          "(Criterio 1 de la rúbrica).")
    return None

# 2) Comparativa de rendimiento del pipeline ---------------------------------
def comparar_pipelines(loader_tf=None, loader_torch=None, epocas=1):
    """TODO (estudiante): medir y comparar el tiempo de carga por época de ambos
    frameworks (con y sin prefetch/cache) y graficarlo."""
    print("TODO: comparar tiempos de carga por época (tf.data vs DataLoader).")
    return {"tf_data_s": None, "dataloader_s": None}

pipeline_real = construir_pipeline_real()          # stub: devuelve None
_comp = comparar_pipelines()                       # stub: devuelve placeholders
print("Etapa A · andamiaje listo. pipeline_real =", pipeline_real,
      "(stub: el estudiante lo implementa sobre datos reales).")

## Etapa B — Entrenamiento acelerado (GPU + precisión mixta)

Representar las **activaciones** en 16 bits (FP16/bfloat16) reduce la VRAM de esos tensores a la mitad y acelera el cómputo al activar los *Tensor Cores*:

$$ \text{VRAM}_{16}^{\text{activaciones}} \approx \tfrac{1}{2}\,\text{VRAM}_{32}^{\text{activaciones}} \qquad \text{aceleración} = \frac{t_{32}}{t_{16}} $$

> 💡 **Matiz importante**
> El ahorro **total** es *menor* que la mitad: la precisión mixta automática (**AMP**) mantiene una copia de los **pesos maestros en FP32** para la actualización del optimizador. En GPUs Ampere+ (L4/A100) suele preferirse **bfloat16** (sin `GradScaler`); en la **T4**, `float16` con escalado de pérdida.

Se proveen, **listas para usar**: (1) una utilidad de ***timing*** por bloque y (2) utilidades de ***checkpointing* a Drive** (guardar/cargar estado). Se deja como `# TODO (estudiante)`: **entrenar el modelo final** (ResNet-18 *transfer learning*) con **AMP**, *early stopping* y *checkpoints*, y **reportar el *speed-up*** FP32 vs FP16.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Celda de soporte 4 · Timing y checkpointing a Drive  (FUNCIONAL)
# Qué hace · Provee un cronómetro de contexto (Cronómetro) y utilidades de checkpoint
#            (guardar_checkpoint / cargar_checkpoint) que escriben en una carpeta (Drive en Colab).
# Por qué  · El estudiante invoca estas utilidades en su bucle de entrenamiento (TODO) para
#            medir el speed-up y para que la rama visual sobreviva entre sesiones de Colab.

class Cronometro:
    """Cronometro de contexto: 'with Cronometro('etiqueta') as c: ...' e imprime los segundos.

    Si hay GPU, sincroniza CUDA antes de medir para que el tiempo sea real (no asíncrono)."""
    def __init__(self, etiqueta="bloque"):
        self.etiqueta = etiqueta
    def __enter__(self):
        if TORCH_OK and GPU_OK:
            torch.cuda.synchronize()
        self.t0 = time.perf_counter()
        return self
    def __exit__(self, *exc):
        if TORCH_OK and GPU_OK:
            torch.cuda.synchronize()
        self.segundos = time.perf_counter() - self.t0
        print(f"[timing] {self.etiqueta}: {self.segundos:.3f} s")
        return False

# Carpeta de checkpoints. En Colab, montar Drive y usar p. ej. "/content/drive/MyDrive/centinela_ckpt".
CKPT_DIR = "checkpoints_centinela"
os.makedirs(CKPT_DIR, exist_ok=True)

def guardar_checkpoint(modelo, optimizador, epoca, mejor_val, nombre="ckpt.pt"):
    """Guarda estado del modelo+optimizador. No rompe si torch no está disponible."""
    ruta = os.path.join(CKPT_DIR, nombre)
    if not TORCH_OK or modelo is None:
        print(f"[checkpoint] (omitido) torch/modelo no disponible -> no se escribe {ruta}.")
        return None
    torch.save({"epoca": epoca, "mejor_val": mejor_val,
                "modelo": modelo.state_dict(),
                "optimizador": optimizador.state_dict() if optimizador else None}, ruta)
    print(f"[checkpoint] guardado en {ruta} (época {epoca}, mejor_val {mejor_val}).")
    return ruta

def cargar_checkpoint(modelo, optimizador=None, nombre="ckpt.pt"):
    """Carga estado si el archivo existe; devuelve la época o None. Robusto ante ausencia."""
    ruta = os.path.join(CKPT_DIR, nombre)
    if not TORCH_OK or modelo is None or not os.path.exists(ruta):
        print(f"[checkpoint] no se carga ({ruta} ausente o sin torch/modelo).")
        return None
    # weights_only=False (explícito): un checkpoint de ENTRENAMIENTO incluye el estado del
    # optimizador y otros objetos Python, que requieren el deserializador completo de torch.
    # Para cargar SOLO los pesos de INFERENCIA se usa weights_only=True (más seguro), como en el lab 06.
    estado = torch.load(ruta, map_location=DEVICE, weights_only=False)
    modelo.load_state_dict(estado["modelo"])
    if optimizador and estado.get("optimizador"):
        optimizador.load_state_dict(estado["optimizador"])
    print(f"[checkpoint] cargado de {ruta} (época {estado['epoca']}).")
    return estado["epoca"]

# Demostración (FUNCIONAL): el cronómetro mide un bloque real, el checkpoint se omite con grace.
with Cronometro("demo · pasada por las imágenes sintéticas"):
    _ = X_img.mean()
guardar_checkpoint(modelo=None, optimizador=None, epoca=0, mejor_val=None)
print("Utilidades de soporte listas: Cronómetro, guardar_checkpoint, cargar_checkpoint.")

### `# TODO (estudiante)` · Entrenamiento con AMP, *early stopping* y *checkpoints*

> 📝 **Tarea del estudiante (Etapa B)**
> Aquí está el trabajo evaluable de entrenamiento acelerado (**Criterio 2 de la rúbrica · 20 %**). Sobre el **modelo final** y los **datos reales**, se debe:
>
> 1. **Cargar un modelo preentrenado** (p. ej. `torchvision.models.resnet18(weights=...)`) adaptando la cabeza al nº de clases (*transfer learning*).
> 2. **Entrenar con precisión mixta (AMP):** `torch.autocast("cuda", dtype=torch.float16)` + `torch.amp.GradScaler()`, midiendo la **VRAM antes/después** de activar FP16 y el **tiempo por época**.
> 3. **Implementar *early stopping*** (paciencia sobre la pérdida de validación) y ***checkpointing*** cada N épocas con `guardar_checkpoint(...)` (provisto).
> 4. **Reportar el *speed-up*** $t_{32}/t_{16}$ y, si aparece **CUDA OOM**, documentar la solución (reducir *batch*, *gradient checkpointing*, etc.).
>
> 💡 *Pista (AMP en PyTorch):*
> ```python
> scaler = torch.amp.GradScaler()
> with torch.autocast("cuda", dtype=torch.float16):
>     perdida = criterio(modelo(X), y)
> scaler.scale(perdida).backward(); scaler.step(optim); scaler.update()
> ```

> ⚠️ La celda siguiente trae **stubs** que no rompen la ejecución (devuelven `modelo_final = None` y un historial vacío). El estudiante reemplaza el cuerpo de `entrenar_amp(...)`.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TODO (estudiante) · Etapa B: entrenamiento acelerado con AMP + early stopping + checkpoints
# Reemplazar los stubs por la implementación propia. Los stubs NO rompen la ejecución.

# 1) Modelo final (transfer learning) ----------------------------------------
# Pista (descomentar y completar cuando torchvision este disponible):
# from torchvision import models
# base = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
# base.fc = nn.Linear(base.fc.in_features, len(CLASES))   # adaptar la cabeza al nº de clases
# modelo_final = base.to(DEVICE)
modelo_final = None  # TODO (estudiante): construir el modelo final (ResNet-18 transfer learning)

# 2) Bucle de entrenamiento con precisión mixta (AMP) ------------------------
def entrenar_amp(modelo, dataloaders, epocas=10, lr=1e-3, paciencia=3):
    """TODO (estudiante): entrenar con AMP (autocast + GradScaler), early stopping y
    checkpointing (guardar_checkpoint). Medir VRAM y tiempo por época; comparar FP32 vs FP16
    para reportar el speed-up (Criterio 2 de la rúbrica). Devolver el historial."""
    print("TODO: implementar el entrenamiento con precisión mixta, early stopping y checkpoints "
          "(Criterio 2 de la rúbrica).")
    return {"train_loss": [], "val_loss": [], "val_acc": [], "speedup": None}

# 3) Reporte de speed-up FP32 vs FP16 ----------------------------------------
def reportar_speedup(t_fp32=None, t_fp16=None):
    """TODO (estudiante): calcular y graficar la aceleración t_fp32/t_fp16 y el ahorro de VRAM."""
    print("TODO: medir y reportar el speed-up FP32 vs FP16 y el ahorro de VRAM.")
    return None

# Llamadas de demostración (NO entrenan: solo confirman que el andamiaje no se rompe)
_hist_B = entrenar_amp(modelo_final, dls_img, epocas=1)
_sp = reportar_speedup()
print("Etapa B · andamiaje listo. modelo_final =", modelo_final, "(stub: el estudiante lo implementa).")

## Etapa C — Exportación y despliegue offline (borde)

Antes de llevar la **rama visual** a la tablet se **comprime** el modelo (cuantización post-entrenamiento a INT8, poda o destilación), se **exporta** a un formato portable (**ONNX/LiteRT**) y se **verifica la paridad de inferencia** entre el modelo original y el exportado: un modelo que predice distinto tras exportarse **no es desplegable**.

$$ \big\| f_{\text{original}}(X) - f_{\text{exportado}}(X) \big\|_\infty < \texttt{atol} \quad (\texttt{atol}=10^{-4}) $$

Se proveen, **listos para usar**: (1) un ***benchmark* de inferencia** (latencia y tamaño en disco) y (2) una **plantilla ONNX con un modelo *dummy* FUNCIONAL** que exporta, recarga con `onnxruntime` y **comprueba la paridad** de extremo a extremo (si `onnx`/`onnxruntime` están instalados; si no, avisa sin romper). Se deja como `# TODO (estudiante)`: exportar el **modelo entrenado**, **cuantizarlo**, validar la **paridad** y medir **latencia/tamaño** antes/después.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Celda de soporte 5 · Benchmark de inferencia + plantilla ONNX (dummy)  (FUNCIONAL)
# Qué hace · (1) Mide latencia (ms/imagen) y tamaño en disco (MB) de un modelo.
#            (2) Exporta un modelo DUMMY a ONNX, lo recarga con onnxruntime y verifica la PARIDAD
#                con np.allclose(atol=1e-4) — todo de punta a punta, sin datos reales.
# Por qué  · Aisla la MECÁNICA de benchmark+exportación+paridad; el estudiante luego repite el
#            patrón con su MODELO ENTRENADO y su modelo CUANTIZADO.

def tamano_mb(ruta):
    """Tamaño de un archivo en MB (0 si no existe)."""
    return os.path.getsize(ruta) / (1024 ** 2) if os.path.exists(ruta) else 0.0

def benchmark_latencia(funcion_inferencia, entrada, repeticiones=50, calentamiento=5):
    """Mide la latencia media (ms/imagen) de 'funcion_inferencia(entrada)'.

    'funcion_inferencia' es cualquier callable que reciba la entrada y devuelva la salida
    (modelo torch, sesión onnxruntime envuelta, intérprete LiteRT, etc.)."""
    for _ in range(calentamiento):
        funcion_inferencia(entrada)
    t0 = time.perf_counter()
    for _ in range(repeticiones):
        funcion_inferencia(entrada)
    ms = (time.perf_counter() - t0) / repeticiones * 1000.0
    print(f"[benchmark] latencia media: {ms:.3f} ms/inferencia ({repeticiones} repeticiones)")
    return ms

# --- Plantilla ONNX con un modelo DUMMY funcional (si hay torch) ---
def demo_exportar_onnx_dummy():
    """Exporta un modelo dummy a ONNX, lo recarga y verifica la paridad. Robusto: avisa sin romper."""
    if not TORCH_OK:
        print("[ONNX] torch no disponible -> se omite la demostración (no rompe el cuaderno).")
        return None
    modelo_dummy = nn.Sequential(nn.Flatten(), nn.Linear(3 * IMG * IMG, len(CLASES))).to(DEVICE).eval()
    X = torch.randn(1, 3, IMG, IMG, device=DEVICE)
    with torch.no_grad():
        salida_torch = modelo_dummy(X).cpu().numpy()
    ruta = os.path.join(CKPT_DIR, "dummy.onnx")
    # Export robusto (mismo patrón que el lab 06): dynamo=False fuerza el exportador estable
    # (TorchScript), que solo necesita "onnx". En torch >=2.9 evita el UserWarning/dependencia
    # de onnxscript del exportador moderno. En torch antiguo el kwarg dynamo no existe -> se
    # reintenta sin el (except TypeError).
    def _exportar(**extra):
        torch.onnx.export(
            modelo_dummy, X, ruta,
            input_names=["entrada"], output_names=["salida"],
            dynamic_axes={"entrada": {0: "lote"}, "salida": {0: "lote"}},
            opset_version=17, **extra)
    try:
        try:
            _exportar(dynamo=False)
        except TypeError:
            _exportar()                                  # torch antiguo: sin el kwarg dynamo
        print(f"[ONNX] modelo dummy exportado a {ruta} · tamaño {tamano_mb(ruta):.3f} MB")
    except Exception as e:
        print("[ONNX] aviso: no se pudo exportar (¿falta torch.onnx?) ->", e)
        return None
    try:
        import onnxruntime
        sess = onnxruntime.InferenceSession(ruta, providers=["CPUExecutionProvider"])
        salida_onnx = sess.run(None, {"entrada": X.cpu().numpy()})[0]
        paridad = np.allclose(salida_torch, salida_onnx, atol=1e-4)
        print(f"[ONNX] PARIDAD original vs ONNX (atol=1e-4): {paridad}")
        # Benchmark de la sesión ONNX (mecánica reutilizable por el estudiante)
        benchmark_latencia(lambda x: sess.run(None, {"entrada": x.cpu().numpy()}), X)
        return paridad
    except Exception as e:
        print("[ONNX] aviso: onnxruntime no disponible -> se omite la verificación de paridad ->", e)
        return None

_paridad_demo = demo_exportar_onnx_dummy()
print("Utilidades de soporte listas: tamano_mb, benchmark_latencia, demo_exportar_onnx_dummy.")

> ⚠️ **Reproducibilidad y paridad de inferencia — la verificación que cierra el ciclo**
> Un modelo que, tras exportarse o cuantizarse, **predice distinto** del original **no es desplegable**. Por eso la exportación se cierra SIEMPRE con una comprobación de **paridad** (`np.allclose(salida_original, salida_exportada, atol=1e-4)`) sobre el mismo lote de entrada y en **modo evaluación** (`modelo.eval()`). La cuantización INT8 introduce un pequeño error esperado: el entregable es la **tabla antes/después** (tamaño, latencia y exactitud) que justifica el *trade-off* para el uso **offline en campo**. Mantén la **semilla** fija y un único *framework* activo al exportar (cargar TF y PyTorch a la vez puede precipitar un **CUDA OOM**).

### `# TODO (estudiante)` · Exportar, cuantizar y validar paridad

> 📝 **Tarea del estudiante (Etapa C)**
> Aquí está el trabajo evaluable de Edge + exportación (**Criterio 3 de la rúbrica · 20 %**). Sobre el **modelo entrenado** de la Etapa B, se debe:
>
> 1. **Comprimir** con al menos una técnica: **cuantización** (post-entrenamiento a INT8, la vía más directa para LiteRT), **poda** de pesos de baja magnitud o **destilación** a un modelo menor.
> 2. **Exportar** a **≥ 2 formatos estándar** (p. ej. `.pth`/`.keras` + **ONNX**; y para el borde, **LiteRT**/`.tflite`).
> 3. **Validar la paridad de inferencia** con `np.allclose(..., atol=1e-4)` (reusando el patrón de la celda de soporte) entre el modelo original y cada formato exportado.
> 4. **Medir latencia y tamaño** antes/después y completar la **tabla comparativa** (tamaño MB · latencia ms · exactitud). Discutir el *trade-off*.
>
> 💡 *Pista (LiteRT/ONNX):* la conversión a LiteRT puede partir del modelo Keras/TF sin pasar por ONNX. La exportación de capas **LSTM/GRU a LiteRT es poco fiable** → por eso la rama temporal se sirve por **API REST en la nube** (decisión de despliegue, Criterio 4).

> ⚠️ La celda siguiente trae **stubs** que no rompen la ejecución (devuelven `None`). El estudiante reemplaza el cuerpo de `exportar_y_cuantizar(...)` y completa la tabla.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TODO (estudiante) · Etapa C: exportar + cuantizar + validar paridad + medir latencia/tamaño
# Reemplazar los stubs por la implementación propia. Los stubs NO rompen la ejecución.

# 1) Exportación y cuantización del modelo ENTRENADO -------------------------
def exportar_y_cuantizar(modelo, entrada_ejemplo=None, formatos=("onnx", "tflite")):
    """TODO (estudiante): poner modelo.eval(); exportar a >=2 formatos (p. ej. ONNX y LiteRT),
    aplicar cuantización (INT8) y devolver las rutas. Reusar la PLANTILLA ONNX de la celda
    de soporte 5 como base (Criterio 3 de la rúbrica)."""
    print("TODO: exportar a ONNX/LiteRT y cuantizar el modelo entrenado (Criterio 3 de la rúbrica).")
    return {}

# 2) Verificación de PARIDAD entre original y exportado ----------------------
def verificar_paridad(salida_original=None, salida_exportada=None, atol=1e-4):
    """TODO (estudiante): comprobar np.allclose(salida_original, salida_exportada, atol=1e-4)
    sobre el MISMO lote, en modo eval. Devolver True/False."""
    print("TODO: verificar la paridad de inferencia con np.allclose(atol=1e-4).")
    return None

# 3) Tabla comparativa antes/después (tamaño · latencia · exactitud) ---------
def tabla_edge_antes_despues():
    """TODO (estudiante): completar la tabla con métricas reales del modelo original (FP32)
    y el optimizado (INT8). Usar tamano_mb(...) y benchmark_latencia(...) provistos."""
    print("TODO: completar la tabla Edge antes/después (tamaño MB · latencia ms · exactitud).")
    filas = [
        ("Tamaño del modelo (MB)",            None, None),
        ("Latencia de inferencia (ms/img)",   None, None),
        ("Exactitud (accuracy/F1 en test)",   None, None),
    ]
    print(f"{'Métrica':<34}{'Antes (FP32)':>14}{'Después (INT8)':>16}")
    for met, antes, despues in filas:
        print(f"{met:<34}{str(antes):>14}{str(despues):>16}")
    return filas

# Llamadas de demostración (NO exportan el modelo real: confirman que el andamiaje no se rompe)
_rutas = exportar_y_cuantizar(modelo_final)        # stub: devuelve {}
_par = verificar_paridad()                          # stub: devuelve None
_tabla = tabla_edge_antes_despues()                 # stub: imprime la tabla con placeholders
print("Etapa C · andamiaje listo (stub: el estudiante exporta, cuantiza y valida su modelo).")

## Sección final — Checklist de entrega + rúbrica de la Fase 3

Una vez industrializado el sistema sobre los **datos reales** del proyecto (no sobre estas imágenes sintéticas), la Fase 3 se entrega con estas piezas. La calificación se rige por la **rúbrica oficial** (abajo), cuyos **seis pesos suman 100 %**.

**Checklist de entrega**

- [ ] **Notebook** (`.ipynb`) **ejecutado en Colab** con todas las salidas visibles: pipeline en ambos *frameworks*, entrenamiento en GPU con AMP, optimización Edge y verificación de exportación.
- [ ] **Pesos entrenados + modelo exportado** en ≥ 2 formatos estándar (**ONNX/LiteRT**, p. ej. `.pth`/`.keras` + `.onnx` + `.tflite`).
- [ ] **Benchmark de latencia/tamaño** (tabla Edge antes/después: tamaño MB · latencia ms · exactitud) con verificación de **paridad** (`np.allclose`, `atol=1e-4`).
- [ ] **Informe técnico de despliegue** (PDF, **máx. 4 páginas**): comparativa de pipelines, rendimiento GPU, tabla de optimización Edge y **ruta de despliegue** (rama visual offline con LiteRT + rama temporal por API REST) con su justificación de ingeniería y citación APA.
- [ ] **Bitácora de IA**: qué se pidió, qué devolvió la IA y qué se **aceptó / modificó / rechazó** y por qué.
- [ ] **Auditoría a la IA**: al menos **un caso real** en que la IA falló (p. ej. `dynamo=True` como única vía de exportación, un pipeline sin `prefetch`, un manejo incorrecto de la VRAM) y cómo se corrigió.
- [ ] Repositorio en **GitHub** o carpeta comprimida `Apellido_Nombre_Centinela_Fase3.zip` con todo lo anterior.

**Rúbrica oficial de la Fase 3** (autoritativa; los seis pesos suman 100 %):

| Criterio (peso) | Qué evidencia |
|---|---|
| **1. Pipeline + Augmentation (20 %)** | El **mismo** pipeline en `tf.data` y `DataLoader` con `prefetch`/`cache`, **≥ 4 aumentos justificados** y comparativa de rendimiento, con código modular y partición sin fuga. |
| **2. Entrenamiento GPU + gestión de recursos (20 %)** | **Precisión mixta** (FP16/bfloat16) con ahorro de VRAM y aceleración documentados; *checkpointing* robusto; **CUDA OOM** resuelto con evidencia; curvas claras. |
| **3. Optimización Edge + exportación (20 %)** | Técnica de compresión (cuantización/poda/destilación) con **tabla antes/después** (tamaño, latencia, precisión) y *trade-off*; exporta en hasta 3 formatos con **verificación ONNX** (`np.allclose`, `atol=1e-4`). |
| **4. Ruta de despliegue + justificación de ingeniería (20 %)** | Ruta completa (**rama visual offline con LiteRT + rama temporal por API REST**); justifica *trade-offs* VRAM/batch/velocidad y elección de *framework*/formato; **Transparencia** y **Beneficio Social** con mitigación de FP/FN. |
| **5. Uso responsable de IA (10 %)** | **Bitácora** con decisiones deliberadas (casos *Modificada/Rechazada* fundamentados); defensa de cada decisión como propia; **auditoría** con un fallo no trivial corregido. |
| **6. Comunicación (informe técnico) (10 %)** | Informe riguroso con comparativa de pipelines, rendimiento GPU, tabla de optimización Edge y ruta de despliegue; narrativa clara y **citación APA** correcta. |

**Total de la Fase 3: 25 % del curso · 40 h · individual (cierra el proyecto integrador).**

> ⚠️ **Este andamiaje NO se entrega tal cual**
> Este cuaderno es punto de partida y trabaja con **datos sintéticos**. La entrega evaluable debe **industrializar el sistema real** de las Fases 1-2 (mismo cliente/problema), e incluir los pesos, el modelo exportado y cuantizado, el *benchmark*, el informe, la **Bitácora de IA** y la **Auditoría a la IA**. Reutilizar este scaffold sin adaptarlo **no satisface** la actividad.

### 🔗 Relacionado
- **Fase 2 (anterior):** Actividad 2 · "Sistema Multimodal Especializado" — el sistema CNN + RNN + fusión que aquí se industrializa.
- **Documento maestro:** Proyecto Integrador *Centinela* (especificación longitudinal de las tres fases, política de datos e integridad de IA).
- **Plantillas de integridad:** Bitácora de IA · Auditoría a la IA (registro de lo pedido / aceptado / modificado / rechazado y del fallo corregido).
- **Material publicado (GitHub Pages):** https://jotamao1985.github.io/Deep_Learning_Usta/
- **Base teórica:** Módulo 3 — Frameworks (TF/PyTorch/Keras 3) · HPC y GPUs · Pipelines y despliegue.